# Accelerate Qwen3-8B with Speculative Decoding and Efficient Draft Models

[Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B) is part of the latest [Qwen family](https://qwenlm.github.io/blog/qwen3/), trained with explicit agentic capabilities. It supports tool invocation, multi-step reasoning, and long context, making it well-suited for agent workflows. Integrated with agentic frameworks such as Hugging Face SmolAgents and QwenAgent, it enables a wide range of agentic applications involving tool calling and reasoning

In this notebook we will demonstrate how to speedup Qwen3-8B inference with speculative decoding using OpenVINO GenAI library.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Generation with OpenVINO GenAI](#Generation-with-OpenVINO-GenAI)
- [Accelerated Generation with Speculative Decoding](#Accelerated-Generation-with-Speculative-Decoding)
- [Further Accelerate Qwen3-8B by Draft Layer Pruning ](#Further-Accelerate-Qwen3-8B-by-Draft-Layer-Pruning)
- [Compute Average Speedup Gain of Speculative Decoding with Qwen3-8B model](#Compute-Average-Speedup-Gain-of-Speculative-Decoding-with-Qwen3-8B-model)



## Prerequisites

> Note: we recommend running this notebook in a virtual environment. 

Install required dependencies

In [ ]:
!pip install -q -r ./smolagents/requirements.txt

## Generation with OpenVINO GenAI
We can simply run generation with the Qwen3-8B model using OpenVINO GenAI library.
[OpenVINO™ GenAI](https://github.com/openvinotoolkit/openvino.genai) is a library of the most popular Generative AI model pipelines, optimized execution methods, and samples that run on top of highly performant OpenVINO Runtime.
This library is friendly to PC and laptop execution, and optimized for resource consumption. It requires no external dependencies to run generative models as it already includes all the core functionality (e.g. tokenization via openvino-tokenizers).

### Prepare Model

First we will download Qwen3-8B model converted to the OpenVINO™ IR (Intermediate Representation) format with weights compressed to INT4 by [NNCF](https://github.com/openvinotoolkit/nncf), available in the OpenVINO LLM collection [Qwen3-8b-int4-ov](https://huggingface.co/OpenVINO/Qwen3-8B-int4-ov ):

In [ ]:
from pathlib import Path
import huggingface_hub as hf_hub

model_id = "OpenVINO/Qwen3-8B-int4-ov"
model_path = Path(model_id.split("/")[-1])

if not model_path.exists():
    hf_hub.snapshot_download(model_id, local_dir=model_path)

### Instantiate a pipeline with OpenVINO Generate API

We will use [OpenVINO Generate API](https://github.com/openvinotoolkit/openvino.genai/blob/master/src/README.md) to create pipelines to run an inference with OpenVINO Runtime. 

Firstly we need to create a pipeline with `LLMPipeline`. `LLMPipeline` is the main object used for text generation using LLM in OpenVINO GenAI API. You can construct it straight away from the folder with the downloaded model. We will provide directory with model and device for `LLMPipeline`. Additionally we provide `SchedulerConfig` for resource management.  Then we can run `generate` method and get the output in text format.

In [ ]:
import openvino_genai as ov_genai
import time


def streamer(subword):
    print(subword, end="", flush=True)
    return False


# select device for inference
device = "GPU"

# define scheduler
scheduler_config = ov_genai.SchedulerConfig()
scheduler_config.num_kv_blocks = 200
scheduler_config.dynamic_split_fuse = False
scheduler_config.max_num_batched_tokens = 8192
scheduler_config.enable_prefix_caching = False
scheduler_config.use_cache_eviction = False

# create a pipeline for generation
pipe = ov_genai.LLMPipeline(model_path, device, scheduler_config=scheduler_config)


After instantiating the pipeline we are ready to generate with the model.

We can configure parameters for decoding. We can create the default config with `ov_genai.GenerationConfig()`, setup parameters, and apply the updated version with `set_generation_config(config)` or put config directly to `generate()`. Since our prompt is already formatted in the Qwen3 chat-template format, we set the generation-config 'apply_chat_template' parameter to 'False'.
To get a more accurate measurement of the generation time, we add a warmup generation step before the actual generation to let the model allocate memory and compile any kernels it needs to reach its full potential.

In [ ]:
import time

generation_config = ov_genai.GenerationConfig()
generation_config.apply_chat_template = False

input_prompt = """<|im_start|>user
In one sentence, explain what blockchain is.
<|im_end|>
<|im_start|>assistant
"""

# We will first do a short warmup to the model so the time measurement will not include the warmup overhead.
generation_config.max_new_tokens = 100
pipe.generate(input_prompt, generation_config)

# Now we can measure the time and see the result
generation_config.max_new_tokens = 2048

start = time.perf_counter()
result = pipe.generate([input_prompt], generation_config, streamer)
# we don't include TTFT in speedup measurement
ar_gen_time = time.perf_counter() - start - (result.perf_metrics.get_ttft().mean / 1000)

In [ ]:
print(f"Generation took {ar_gen_time:.3f} seconds")

import gc

# del pipe
# gc.collect()

## Accelerated Generation with Speculative Decoding
Speculative decoding is a lossless decoding paradigm introduced in a recent [ICML paper](https://arxiv.org/abs/2211.17192) for accelerating auto-regressive generation with LLMs.
The method aims to mitigate the inherent latency bottleneck caused by the sequential nature of auto-regressive generation.
Speculative decoding employs a draft language model to generate a block of \(\gamma\) candidate tokens.
The LLM, referred to as the target model, then processes these candidate tokens in parallel.
The algorithm examines each token's probability distribution, calculated by both the target and draft models, to determine whether the token should be accepted or rejected.

In this section we will demonstrate how to accelerate the generation of Qwen3-8B using speculative-decoding, with the open source Qwen3-0.6B model as the draft model.

The [Qwen3‑0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) model is a compact yet capable language model in Alibaba’s Qwen3 series, featuring just 0.6 billion parameters (~0.44B non-embedding), 28 layers, and a 32K token context window, making it ideal for edge or low-resource deployments. Despite its small size, it's built atop the same hybrid reasoning architecture as its larger siblings, supporting both thinking mode (for logical reasoning, math, and code) and non-thinking mode (for fast, conversational responses) within a unified framework.

We will first download the draft-model and then we will use it to initialize a speculative-decoding generation pipeline:

In [ ]:
from pathlib import Path
import huggingface_hub as hf_hub

draft_model_id = "OpenVINO/Qwen3-0.6B-int8-ov"
draft_model_path = Path(draft_model_id.split("/")[-1])

if not draft_model_path.exists():
    hf_hub.snapshot_download(draft_model_id, local_dir=draft_model_path)

In [ ]:
# Define schedulers for main/draft models

draft_device = "GPU"
scheduler_config = ov_genai.SchedulerConfig()
scheduler_config.num_kv_blocks = 200
scheduler_config.dynamic_split_fuse = False
scheduler_config.max_num_batched_tokens = 8192
scheduler_config.enable_prefix_caching = False
scheduler_config.use_cache_eviction = False

draft_scheduler_config = ov_genai.SchedulerConfig()
draft_scheduler_config.num_kv_blocks = 200
draft_scheduler_config.dynamic_split_fuse = False
draft_scheduler_config.max_num_batched_tokens = 8192

draft_model = ov_genai.draft_model(draft_model_path, draft_device, scheduler_config=draft_scheduler_config)

# create a pipeline with a draft model for generation with speculative decoding
pipe = ov_genai.LLMPipeline(model_path, device, draft_model=draft_model, scheduler_config=scheduler_config)

Now we are ready to generate with our speculative decoding pipeline. We will run a small warmup step before measuring the actual generation time

In [ ]:
# Define in the generation config the numbers of candidates generated by draft_model per iteration
generation_config.num_assistant_tokens = 3
generation_config.apply_chat_template = False

# Again we will do a short warmup before measuring time for the model
generation_config.max_new_tokens = 100
pipe.generate(input_prompt, generation_config)

# Now we can measure the time and see the result
generation_config.max_new_tokens = 2048

start = time.perf_counter()
result = pipe.generate([input_prompt], generation_config, streamer)
# we don't include TTFT in speedup measurement
sd_gen_time = time.perf_counter() - start - (result.perf_metrics.get_ttft().mean / 1000)

In [ ]:
print(f"Generation took {sd_gen_time:.3f} seconds")

Let's calculate the speedup achieved when accelerating using Qwen3-0.6B as a draft for the specific example we used:

In [ ]:
print(f"End to end speedup with speculative decoding is {ar_gen_time / sd_gen_time:.2f}x")

## Further Accelerate Qwen3-8B by Draft Layer Pruning 

Layer pruning is a model compression technique for large language models (LLMs) that reduces inference cost by removing entire transformer layers from the network. We leveraged this technique to generate a smaller yet qualitative draft-model - [pruned-qwen3-draft](https://huggingface.co/OpenVINO/Qwen3-pruned-6L-from-0.6B-int8-ov), eventually further accelerating the speculative-decoding generation of Qwen3-8B model.

We will download the enhanced draft-model and repeat the measurement in previous section to demonstrate the speedup improvement:


In [ ]:
from pathlib import Path
import huggingface_hub as hf_hub

draft_model_id = "OpenVINO/Qwen3-pruned-6L-from-0.6B-int8-ov"
draft_model_path = Path(draft_model_id.split("/")[-1])

if not draft_model_path.exists():
    hf_hub.snapshot_download(draft_model_id, local_dir=draft_model_path)

In [ ]:
# Define schedulers for main/draft models

draft_device = "GPU"
scheduler_config = ov_genai.SchedulerConfig()
scheduler_config.num_kv_blocks = 200
scheduler_config.dynamic_split_fuse = False
scheduler_config.max_num_batched_tokens = 8192
scheduler_config.enable_prefix_caching = False
scheduler_config.use_cache_eviction = False

draft_scheduler_config = ov_genai.SchedulerConfig()
draft_scheduler_config.num_kv_blocks = 200
draft_scheduler_config.dynamic_split_fuse = False
draft_scheduler_config.max_num_batched_tokens = 8192

draft_model = ov_genai.draft_model(draft_model_path, draft_device, scheduler_config=draft_scheduler_config)

# create a pipeline with a draft model for generation with speculative decoding
pipe = ov_genai.LLMPipeline(model_path, device, draft_model=draft_model, scheduler_config=scheduler_config)

In [ ]:
# We need to define in the generation config how many tokens the draft should predict in each cycle
generation_config.num_assistant_tokens = 3
generation_config.apply_chat_template = False

# Again we will do a short warmup before measuring time for the model
generation_config.max_new_tokens = 100
pipe.generate(input_prompt, generation_config)

# Now we can measure the time and see the result
generation_config.max_new_tokens = 2048

start = time.perf_counter()
result = pipe.generate([input_prompt], generation_config, streamer)
accelerated_sd_gen_time = time.perf_counter() - start - (result.perf_metrics.get_ttft().mean / 1000)  # we don't include TTFT in speedup measurement

In [ ]:
print(f"Generation took {accelerated_sd_gen_time:.3f} seconds")
print(f"End to end speedup with speculative decoding is {ar_gen_time / accelerated_sd_gen_time:.2f}x")

## Compute Average Speedup Gain of Speculative Decoding with Qwen3-8B model

In this section we measure the average speedup gain of speculative-decoding with Qwen3-8B model over multiple examples. 
We use a small mix of summarization, reasoning, math and classification examples for the speedup evaluation. The examples are taken from the [CNN/DailyMail](https://huggingface.co/datasets/abisee/cnn_dailymail) and the [MT-Bench](https://huggingface.co/datasets/philschmid/mt-bench) datasets.  
We run the model on these examples twice: first without speculative-decoding, then with speculative-decoding. We compute the total time ratio of the two methods- which is the speedup, and finally we compute the average speedup over all tested examples.

### 1. Run target model without speculative decoding
We will first run generation without speculative-decoding, but this time we will run it over multiple examples. The examples are provided in a json file.

In [ ]:
import openvino_genai as ov_genai
import sys
import time
from tqdm import tqdm

print(f"Loading model from {model_path}")

# Define scheduler
scheduler_config = ov_genai.SchedulerConfig()
scheduler_config.num_kv_blocks = 200
scheduler_config.dynamic_split_fuse = False
scheduler_config.max_num_batched_tokens = 8192
scheduler_config.enable_prefix_caching = False
scheduler_config.use_cache_eviction = False

pipe = ov_genai.LLMPipeline(model_path, device, scheduler_config=scheduler_config)

generation_config = ov_genai.GenerationConfig()
generation_config.apply_chat_template = False

print("Loading prompts...")
import json

f = open("prompts.json")
prompts = json.load(f)

# We will first do a short warmup to the model so the time measurement will not include the warmup overhead.
generation_config.max_new_tokens = 100
pipe.generate("This is a warmup prompt", generation_config)

# finished warmup step, let's run our examples
generation_config.max_new_tokens = 2048
times_auto_regressive = []
print("Running Auto-Regressive generation...")
for prompt in tqdm(prompts):
    start_time = time.perf_counter()
    result = pipe.generate(prompt, generation_config)
    end_time = time.perf_counter()
    times_auto_regressive.append(end_time - start_time)
print("Done")

import gc

del pipe
gc.collect()

### 2. Run target model with speculative decoding
Now we will run generation with speculative-decoding over the same examples. 

In the following dropdown list you can choose which draft to use for the speculative-decoding pipeline:

In [ ]:
import ipywidgets as widgets

draft_model_name = widgets.Dropdown(
    options=["Qwen3-0.6B-int8-ov", "Qwen3-0.6B-pruned-int8-ov"],
    value="Qwen3-0.6B-int8-ov",  # default value
    description="Select Draft Model:",
)

draft_model_name

In [ ]:
import openvino_genai as ov_genai
import time
from tqdm import tqdm

draft_model_path = draft_model_name.value

print("Loading prompts...")
import json

f = open("prompts.json")
prompts = json.load(f)

# Define scheduler
scheduler_config = ov_genai.SchedulerConfig()
scheduler_config.num_kv_blocks = 200
scheduler_config.dynamic_split_fuse = False
scheduler_config.max_num_batched_tokens = 8192
scheduler_config.enable_prefix_caching = False
scheduler_config.use_cache_eviction = False
# Define scheduler for the draft

draft_scheduler_config = ov_genai.SchedulerConfig()
draft_scheduler_config.num_kv_blocks = 200
draft_scheduler_config.dynamic_split_fuse = False
draft_scheduler_config.max_num_batched_tokens = 8192

draft_model = ov_genai.draft_model(draft_model_path, device, scheduler_config=draft_scheduler_config)

pipe = ov_genai.LLMPipeline(model_path, device, draft_model=draft_model, scheduler_config=scheduler_config)

generation_config = ov_genai.GenerationConfig()
generation_config.num_assistant_tokens = 3
generation_config.apply_chat_template = False

# Again, We will first do a short warmup
generation_config.max_new_tokens = 100
pipe.generate("This is a warmup prompt", generation_config)

# finished warmup step, let's run our examples
generation_config.max_new_tokens = 2048
times_speculative_decoding = []

print("Running Speculative Decoding generation...")
for prompt in tqdm(prompts):
    start_time = time.perf_counter()
    result = pipe.generate(prompt, generation_config)
    end_time = time.perf_counter()
    times_speculative_decoding.append(end_time - start_time)
print("Done")

### 3. Calculate average speedup


In [ ]:
avg_speedup = sum([x / y for x, y in zip(times_auto_regressive, times_speculative_decoding)]) / len(prompts)
print(f"average speedup: {avg_speedup:.2f}")